### CareBridge — train the Dagbani voice (Track B)

Trains a Piper/VITS Dagbani voice on the licensed corpus, then exports the ONNX
pack the app loads. Everything here is the repository's own
`tool/track_b/` scripts, embedded verbatim by `make_colab_notebook.py` — do not
edit the cells below, edit the scripts in the repo and regenerate.

- **Data**: `fiifinketia/dagbani-bible-audio-text-tts`, CC-BY-4.0, ~137 h at 24 kHz.
- **Warm start**: `_base_model/base_model.ckpt` (933 MB) holds the vocoder, so only
  the text side learns from scratch.
- **Runtime**: expect 6–12 h on an L4. Checkpoints live in Google Drive, so a
  disconnected session does not lose the run — it costs ~4 GB of Drive space.
- **Do not commit the resulting weights** to the repo until the licence is settled;
  see `assets/tts/dagbani_piper/PROVENANCE.md`.

Set the two knobs in the next cell, then Run all.


In [ ]:
# Colab -> Drive is the only place a checkpoint survives a session ending.
# A T4/L4 run of this length WILL be disconnected at some point.
from google.colab import drive
import os
drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/carebridge-dagbani"
os.makedirs(WORK_DIR, exist_ok=True)
print("checkpoints will survive in:", WORK_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
EPOCHS = 60      # 20 gives a listening sample sooner; 60 is the real run
BATCH = 32
SHARDS = 37      # 37 shards is the whole licensed split; 3 is a smoke test
MOS = "off"      # the UTMOS predictor needs egress; leaving it on aborts epoch 1

APP_LINES_TEXT = r"""
Naafo
Bɔ ni ti shɛli?
M bo ni a daa
Tin yaa
Ee
Ayi
Mi bɛ mi
Dii
Yihi
Lahi
Naɣisi
Sɔŋsi
Yi tiŋ bɛ ni kpeeni pam
Ti sɔŋsi nira maa ni kpeenim laɣim
Tin yaa ka ti maani sɔŋsi tiŋ maa
Bini kpalansi maa ti nu tana
Bini kpalansi maa ɣiri tɔɣi bee ka o gbili
Bini kpalansi maa ti zuɣa
Bini kpalansi maa wum kpibu pam bee ka o ti mɔri
Bini kpalansi maa bini pam bee ka o ti bini pam
Bini kpalansi maa yaa-ŋa bee nini ɣari
Tuli kpamba maa kɔbigu, o pɔŋ, bee o yiɣiri ni bini
Bini kpalansi maa yiɣiri bini maa pam
Bini maa ti ni tooi nu tana bee ka o nu
Bini maa yiɣiri bini maa pam
Bini maa ti zuɣa
Bini maa wum kpibu pam bee ka o ti mɔri
Bini maa ɣiri tɔɣi bee ka o ti ɣiri ti tuhi
Nɔri bɛ bini maa tuhi
Bini maa ti pɔŋ nini pam, bee o ti nɔri pɔŋ
Bini maa ti bini ti pii dabaasi anahi
Naɣa maa kɔbigu pam
Naɣa maa m bɛ laɣim pam ti o gɔri ti ti mali yɛn
Naɣa maa bini pam
Naɣa maa ti kɔbigu taba pam
Naɣa maa ti zuɣa
Bini yiɣiri bɛ bini maa
Ni o bɛ ni tuhi bini, bini maa tuhi ti laɣim ni yili maa
Naɣa maa yiɣiri bini maa pam
Bini kpalansi maa ti nu tana
Bini kpalansi maa ɣiri tɔɣi bee ka o gbili
Bini kpalansi maa ti zuɣa
Bini kpalansi maa wum kpibu pam bee ka o ti mɔri
Bini kpalansi maa bini pam bee ka o ti bini pam
Bini kpalansi maa yaa-ŋa bee nini ɣari
Tuli kpamba maa kɔbigu, o pɔŋ, bee o yiɣiri ni bini
Bini kpalansi maa yiɣiri bini maa pam
Bini maa ti ni tooi nu tana bee ka o nu
Bini maa yiɣiri bini maa pam
Bini maa ti zuɣa
Bini maa wum kpibu pam bee ka o ti mɔri
Bini maa ɣiri tɔɣi bee ka o ti ɣiri ti tuhi
Nɔri bɛ bini maa tuhi
Bini maa ti pɔŋ nini pam, bee o ti nɔri pɔŋ
Bini maa ti bini ti pii dabaasi anahi
Naɣa maa kɔbigu pam
Naɣa maa m bɛ laɣim pam ti o gɔri ti ti mali yɛn
Naɣa maa bini pam
Naɣa maa ti kɔbigu taba pam
Naɣa maa ti zuɣa
Bini yiɣiri bɛ bini maa
Ni o bɛ ni tuhi bini, bini maa tuhi ti laɣim ni yili maa
Naɣa maa yiɣiri bini maa pam
Bini maa nyɛla din yaa. Ti maani sɔŋsi tiŋ maa ka o nu tana.
N ni nya bini shɛŋa n-yɛliya.
Mi bɛ mi bini shɛŋa ŋɔ.
N yɛn yɛliya bini kam ni daa piligi shɛm.
Yi tiŋ bɛ ni kpeeni pam
Yi tiŋ bɛ ni kpeeni pam
Yi tiŋ bɛ ni kpeeni pam
Ni bini maa ti niŋ ka o nui tana, yi tiŋ bɛ ni kpeeni pam.
Ni bini maa ti niŋ ka o nui tana, yi tiŋ bɛ ni kpeeni pam.
Ni bini maa ti niŋ ka o nui tana, yi tiŋ bɛ ni kpeeni pam.

"""


In [ ]:
# The must-say sentences the voice has to be able to pronounce; measured against
# these is what decides whether the pack is usable in the app at all.
import os
from pathlib import Path
os.makedirs("/content/track_b", exist_ok=True)
Path("/content/track_b/app_lines.txt").write_text(APP_LINES_TEXT, encoding="utf8")
print(len(Path("/content/track_b/app_lines.txt").read_text(encoding="utf8").splitlines()), "must-say lines")


In [ ]:
%%writefile /content/track_b/fetch_dagbani.py
"""Pull the licensed Dagbani corpus into the training shape the pack pipeline wants.

Input:  fiifinketia/dagbani-bible-audio-text-tts — 53,410 utterances, ~137 h,
        24 kHz mono, CC-BY-4.0 (verified on the hub 2026-10-18).
Output: <out>/audio/*.wav at the sample rate piper-style VITS trains on,
        <out>/metadata.jsonl with {"audio","text","language","speaker"},
        <out>/corpus_stats.json with hours, letter inventory and licence.

Shards are read over HTTP range requests, so `--shards 1` costs a few hundred
MB rather than the 12 GB dataset.

Run:  python tool/track_b/fetch_dagbani.py --out data/dagbani --shards 1
"""
import argparse, collections, io, json, re, statistics, sys
from pathlib import Path

import fsspec
import pyarrow.parquet as pq
import soundfile as sf

DATASET = "fiifinketia/dagbani-bible-audio-text-tts"
LICENSE = "CC-BY-4.0"
SHARDS = 37
RATE = 22050  # what the app's existing VITS packs train at

# Read-speaker bible text still carries chapter markers and verse numbers.
NOISE = re.compile(r"^[0-9]+\s*$")


def normalise(text: str) -> str:
    """One line, no verse numbering, no double spaces, no trailing punctuation
    the model would have to learn to pause on."""
    text = text.replace("\n", " ")
    text = " ".join(w for w in text.split() if not NOISE.match(w))
    return text.strip()


def shard_rows(index: int):
    url = (
        f"https://huggingface.co/datasets/{DATASET}/resolve/main/"
        f"data/train-{index:05d}-of-{SHARDS:05d}.parquet?download=true"
    )
    fs = fsspec.filesystem("http")
    with fs.open(url, "rb", block_size=32 * 2**20) as handle:
        parquet = pq.ParquetFile(handle)
        for group in range(parquet.num_row_groups):
            for row in parquet.read_row_group(group).to_pylist():
                yield row


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/dagbani")
    ap.add_argument("--shards", type=int, default=1, help="how many shards to pull")
    ap.add_argument("--limit", type=int, default=0, help="cap utterances (0 = all)")
    ap.add_argument("--min-seconds", type=float, default=1.2)
    ap.add_argument("--max-seconds", type=float, default=20.0)
    args = ap.parse_args()

    out = Path(args.out)
    (out / "audio").mkdir(parents=True, exist_ok=True)
    metadata = out / "metadata.jsonl"
    stats_path = out / "corpus_stats.json"

    inventory = collections.Counter()
    durations, written, skipped = [], 0, 0
    with metadata.open("w", encoding="utf8") as sink:
        for index in range(min(args.shards, SHARDS)):
            for row in shard_rows(index):
                if args.limit and written >= args.limit:
                    break
                text = normalise(row.get("text") or "")
                audio = row.get("audio")
                if not text or not audio or not audio.get("bytes"):
                    skipped += 1
                    continue
                samples, source_rate = sf.read(io.BytesIO(audio["bytes"]))
                seconds = len(samples) / source_rate
                if not args.min_seconds <= seconds <= args.max_seconds:
                    skipped += 1
                    continue
                name = f"dag_{index:02d}_{written:06d}.wav"
                sf.write(
                    out / "audio" / name,
                    samples,
                    source_rate,
                    subtype="PCM_16",
                )
                inventory.update(text)
                durations.append(seconds)
                sink.write(
                    json.dumps(
                        {
                            "audio": f"audio/{name}",
                            "text": text,
                            "language": "dag",
                            "speaker": "dagbani-bible",
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )
                written += 1
            if args.limit and written >= args.limit:
                break

    letters = "".join(sorted(c for c in inventory if not c.isspace()))
    stats = {
        "dataset": DATASET,
        "licence": LICENSE,
        "utterances": written,
        "skipped": skipped,
        "hours": round(sum(durations) / 3600, 2),
        "meanSeconds": round(statistics.mean(durations), 2) if durations else 0,
        "trainSampleRate": RATE,
        "sourceSampleRate": 24000,
        "letters": letters,
        "letterCount": len(letters),
    }
    stats_path.write_text(json.dumps(stats, indent=2, ensure_ascii=False) + "\n", encoding="utf8")
    print(
        f"{written} utterances ({stats['hours']} h) -> {out}, "
        f"{skipped} skipped, inventory {len(letters)} letters"
    )
    return 0 if written else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/track_b/prepare_symbols.py
"""Turn the fetched Dagbani corpus into what piper1-gpl trains on.

The trainer's `--data.phoneme_type text` mode takes UTF-8 codepoints as the
symbol sequence, so the symbol inventory is ours to fix. This pins it, filters
the corpus to it, and answers the only question that matters before spending
GPU hours: can this data teach a voice that says everything the app needs?

Outputs in --out:
  metadata.csv     piper training CSV:  file.wav|characters
  phonemes.json    pinned symbol -> id map (given to --data.phonemes_path)
  coverage.json    hours, letters, kept/dropped lines, missing letters

Run:  python tool/track_b/prepare_symbols.py --corpus data/dagbani
"""
import argparse, json, re, sys, unicodedata
from pathlib import Path

PAD, BOS, EOS = "_", "^", "$"
# Piper adds these boundaries itself; they must be in the pinned inventory.
SPECIALS = [PAD, BOS, EOS]
APP_STRING_FILES = (
    "lib/core/i18n/dagbani_strings.dart",
    "lib/core/i18n/speech_bank.dart",
)
# The trainer lowercases and NFD-normalises, so the inventory must be built on
# exactly that form or the ids would not line up.
SEPARATORS = (" ", ",", ".", ":", ";", "-", "?", "!", "'")


def app_dagbani_lines(repo: Path, from_file: Path | None) -> list[str]:
    """The sentences the voice must be able to say.

    Read from the app's own string files when the repo is present, otherwise
    from a dump made on the machine that has it (`--app-lines-out`), so this
    gate can also run on a training box that has no checkout.
    """
    if from_file is not None:
        return [
            line.strip()
            for line in from_file.read_text(encoding="utf8").splitlines()
            if line.strip()
        ]
    lines = []
    for rel in APP_STRING_FILES:
        path = repo / rel
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf8").splitlines():
            match = re.search(r"dagbani:\s*['\"](.+)['\"],?\s*$", line)
            if match:
                lines.append(match.group(1))
    return lines


def symbols_of(text: str) -> set[str]:
    folded = unicodedata.normalize("NFD", text.lower())
    return {c for c in folded if not c.isspace()}


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--corpus", default="data/dagbani")
    ap.add_argument("--repo", default=".")
    ap.add_argument("--max-lines", type=int, default=0, help="cap training rows")
    ap.add_argument("--app-lines", help="read the must-say sentences from a file instead of the repo")
    ap.add_argument("--app-lines-out", help="write the must-say sentences for a machine with no checkout")
    args = ap.parse_args()

    corpus = Path(args.corpus)
    rows = [
        json.loads(line)
        for line in (corpus / "metadata.jsonl").read_text(encoding="utf8").splitlines()
        if line.strip()
    ]
    stats = json.loads((corpus / "corpus_stats.json").read_text(encoding="utf8"))
    if not rows:
        print("no metadata.jsonl rows; run fetch_dagbani.py first")
        return 1

    wanted = app_dagbani_lines(Path(args.repo), Path(args.app_lines) if args.app_lines else None)
    if args.app_lines_out:
        Path(args.app_lines_out).write_text("\n".join(wanted) + "\n", encoding="utf8")
    needed = {c for line in wanted for c in symbols_of(line)}
    seen: set[str] = set()
    for row in rows:
        seen |= symbols_of(row["text"])
    # The inventory comes from the audio, not from wish lists: a symbol with no
    # spoken example would train an un pronounceable embedding.
    inventory = sorted(seen | set(SEPARATORS) | set(SPECIALS))
    # Letters the app must say that this slice of the corpus never spoke. They
    # shrink as more shards are fetched, and must be empty before training.
    needed_not_seen = sorted(needed - seen)

    allowed = set(inventory)
    kept, dropped = [], []
    for row in rows:
        folded = unicodedata.normalize("NFD", row["text"].lower())
        if {c for c in folded} - allowed:
            dropped.append(row["audio"])
            continue
        kept.append(row)
    if args.max_lines:
        kept = kept[: args.max_lines]

    phonemes = {symbol: index for index, symbol in enumerate(inventory)}
    (corpus / "phonemes.json").write_text(
        json.dumps(phonemes, ensure_ascii=False, indent=2) + "\n", encoding="utf8")

    with (corpus / "metadata.csv").open("w", encoding="utf8", newline="") as sink:
        for row in kept:
            name = Path(row["audio"]).name
            text = unicodedata.normalize("NFD", row["text"].lower())
            sink.write(f"{name}|{text}\n")

    app_speakable = [line for line in wanted
                     if not ({c for c in symbols_of(line)} - set(phonemes))]
    coverage = {
        "dataset": stats["dataset"],
        "licence": stats["licence"],
        "corpusHours": stats["hours"],
        "trainingRows": len(kept),
        "droppedRows": len(dropped),
        "symbols": len(phonemes),
        "appLines": len(wanted),
        "appLinesSpeakable": len(app_speakable),
        "letters": "".join(s for s in inventory if s not in SPECIALS),
        "neededButUnseen": needed_not_seen,
    }
    (corpus / "coverage.json").write_text(
        json.dumps(coverage, indent=2, ensure_ascii=False) + "\n", encoding="utf8")

    print(
        f"rows {len(kept)} kept / {len(dropped)} dropped | symbols {len(phonemes)} | "
        f"app lines speakable {len(app_speakable)}/{len(wanted)} | {stats['hours']} h"
    )
    unprovable = [line for line in wanted if line not in app_speakable]
    if unprovable:
        print("BLOCKED, corpus cannot teach:", " ".join(sorted(
            {c for line in unprovable for c in symbols_of(line)} - set(phonemes)
        )))
        return 2
    print("ready: every letter the app needs is in the training inventory")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/track_b/colab_train.sh
#!/usr/bin/env bash
# Train a Dagbani voice with the maintained Piper trainer, then export the pack
# the CareBridge app loads. Run this on a GPU box or a Colab L4/L200 notebook;
# the data stage (tool/track_b/fetch_dagbani.py + prepare_symbols.py) already
# ran locally and produced metadata.csv, phonemes.json and audio/.
#
# Usage:  CORPUS=/content/dagbani bash colab_train.sh
#
# Verified against OHF-Voice/piper1-gpl docs/TRAINING.md on 2026-10-18:
#   - text phoneme type:  --data.phoneme_type text, CSV "file.wav|symbols"
#   - pinned inventory:   --data.phonemes_path
#   - warmstart:          --model.vocoder_warmstart_ckpt (allows a new inventory)
#   - export:             python3 -m piper.train.export_onnx
set -euo pipefail

CORPUS="${CORPUS:-/content/dagbani}"
EPOCHS="${EPOCHS:-60}"
BATCH="${BATCH:-32}"
EARS="${EARS:-50}"          # utterances held out for the listening sample
WORK="${WORK:-/content/piper-work}"

apt-get update -qq && apt-get install -y -qq build-essential cmake ninja-build python3-venv >/dev/null

# Data stage, when the corpus was not prepared beforehand. Upload
# fetch_dagbani.py, prepare_symbols.py and app_lines.txt (from
# `prepare_symbols.py --app-lines-out`) alongside this run.
if [[ ! -s "$CORPUS/metadata.csv" ]]; then
  python3 -m pip install -q pyarrow fsspec soundfile requests
  TOOL="${TOOL_DIR:-/content/track_b}"
  python3 "$TOOL/fetch_dagbani.py" --out "$CORPUS" --shards "${SHARDS:-37}"
  python3 "$TOOL/prepare_symbols.py" --corpus "$CORPUS" --app-lines "$TOOL/app_lines.txt"
fi

mkdir -p "$WORK"
cd "$WORK"
if [[ ! -d piper1-gpl ]]; then
  git clone --depth 1 https://github.com/OHF-Voice/piper1-gpl.git
fi
cd piper1-gpl
python3 -m venv .venv
source .venv/bin/activate
python3 -m pip install -q -U pip
python3 -m pip install -q -e '.[train]'
./build_monotonic_align.sh

# 933 MB; holds the vocoder weights, so only the text side learns from scratch.
if [[ ! -f base_model.ckpt ]]; then
  wget -q -O base_model.ckpt \
    "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/_base_model/base_model.ckpt"
fi

SYMBOLS="$(python3 -c "import json;print(len(json.load(open('$CORPUS/phonemes.json'))))")"

# Upstream also keeps a second ModelCheckpoint monitoring "val_mos". If the
# UTMOS predictor cannot be downloaded (no egress), val_mos is never logged and
# current Lightning aborts at the END OF THE FIRST EPOCH rather than warning and
# skipping -- after hours of training. MOS=off drops the predictor and that
# callback. Same reasoning for TORCH_EXPORT_LEGACY: torch>=2.13 defaults
# torch.onnx.export to the dynamo front end, which fails on the flow modules.
MOS_ARGS=""
if [[ "${MOS:-on}" == "off" ]]; then
  python3 - <<'PY'
import pathlib, re
path = pathlib.Path("src/piper/train/__main__.py")
source = path.read_text()
trimmed, count = re.subn(
    r"\n *ModelCheckpoint\(\s*monitor=\"val_mos\",.*?\),\n", "\n", source, flags=re.S)
if count != 1:
    raise SystemExit(f"expected to remove one val_mos checkpoint, removed {count}")
path.write_text(trimmed)
print("disabled UTMOS checkpoint monitoring for this run")
PY
  MOS_ARGS="--model.mos_metric none"
fi
if [[ "${TORCH_EXPORT_LEGACY:-0}" == "1" ]]; then
  python3 - <<'PY'
import pathlib
path = pathlib.Path("src/piper/train/export_onnx.py")
source = path.read_text()
if "dynamo=False" not in source:
    path.write_text(source.replace("opset_version=OPSET_VERSION,",
                                   "opset_version=OPSET_VERSION,\n        dynamo=False,", 1))
print("export will use the legacy ONNX front end")
PY
fi

echo "inventory: $SYMBOLS symbols, epochs $EPOCHS, batch $BATCH"

# --data.espeak_voice is a required CLI argument but is not consulted when
# phoneme_type is text; the Dagbani alphabet is pinned by phonemes.json instead.
python3 -m piper.train fit \
  $MOS_ARGS \
  --data.voice_name dagbani \
  --data.csv_path "$CORPUS/metadata.csv" \
  --data.audio_dir "$CORPUS/audio" \
  --data.phoneme_type text \
  --data.espeak_voice en-us \
  --data.phonemes_path "$CORPUS/phonemes.json" \
  --data.num_symbols "$SYMBOLS" \
  --data.cache_dir "$CORPUS/cache" \
  --data.config_path "$CORPUS/model.onnx.json" \
  --data.batch_size "$BATCH" \
  --model.sample_rate 22050 \
  --model.vocoder_warmstart_ckpt base_model.ckpt \
  --trainer.max_epochs "$EPOCHS" \
  --trainer.devices 1 \
  --trainer.default_root_dir "$WORK/run"

CKPT="$(find "$WORK/run" -name '*.ckpt' -not -name 'base_model.ckpt' | sort | tail -1)"
echo "checkpoint: $CKPT"

python3 -m piper.train.export_onnx \
  --checkpoint "$CKPT" \
  --output-file "$CORPUS/model.onnx"

# Sample what the app must be able to say, straight from the trained model.
python3 -m piper.infer \
  --config "$CORPUS/model.onnx.json" \
  --speech-text "Ti bɔri nyɛ o tɛn'shɛga." \
  --output "$CORPUS/earcheck_1.wav" || \
  echo "piper.infer CLI differs; run eval_voice.py with the exported pack instead"

echo
echo "Done. Copy back: $CORPUS/model.onnx $CORPUS/model.onnx.json"
echo "Then locally:  python tool/track_b/eval_voice.py --pack <dir>"
echo "and:           python tool/build_voice_manifests.py"


In [ ]:
%%bash -s "$EPOCHS" "$BATCH" "$SHARDS" "$MOS" "$WORK_DIR"
# Scripts stay on local disk; only the training root goes to Drive, so a
# disconnected session keeps its checkpoints without copying a 14 GB corpus.
set -eux
CORPUS=/content/dagbani TOOL_DIR=/content/track_b WORK="$5/piper-work" \
  EPOCHS="$1" BATCH="$2" SHARDS="$3" MOS="$4" TORCH_EXPORT_LEGACY=1 \
  bash /content/track_b/colab_train.sh


### After the run

`colab_train.sh` leaves `model.onnx`, `model.onnx.json`, `coverage.json` and
`earcheck_1.wav` in `/content/dagbani`. Copy that folder into
`MyDrive/carebridge-dagbani/final/`, then on this machine:

```
python tool/track_b/eval_voice.py --pack <the folder>   # contract + RTF + listening clips
python tool/build_voice_manifests.py                   # writes model_manifest.json
```

`coverage.json` must read 73/73 before the pack goes anywhere near the app, and a
native Dagbani ear has to sign the accent off — the app's own invariant, and the
reason this is not a number we get to decide alone.
